# Lab 5: Spark Structured APIs

In this class, we will continue to learn about the Spark Structured APIs, including PySpark SQL APIs. 

## 1. About Jupyter Notebook

Enter the IP and your jupyter port in the web browser. For example: `172.18.43.129:11092` and enter the default jupyter password. 

The default jupyter notebook path is `/data/`.

## 2. About [Spark](https://spark.apache.org/docs/latest/)


In [1]:
import pyspark
pyspark.__version__

'4.1.0'

### Apache Spark 4.x 需注意的更新点 
- 放弃 Scala 2.12，将 Scala 2.13 作为默认
- 放弃 JDK 8/11，将 JDK 17 作为默认
- 默认使用 ANSI SQL 模式
- 支持 SQL 用户定义函数
- 将 Pandas 升级到 2
- 放弃 Python 3.8 支持

## 3.复杂JSON数据处理（Optional）

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.executor.memoryOverhead", "4g") \
    .config("spark.sql.shuffle.partitions", "2000") \
    .getOrCreate()

shows = spark.read.json("shows-silicon-valley.json")
shows.count()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/19 16:21:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


1

In [3]:
shows.printSchema()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: string (nullable = true)
 |    |    |    |-- airstamp: string (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- medium: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = true

In [4]:
import pyspark.sql.types as T
import pyspark.sql.functions as F

episode_links_schema = T.StructType(
    [
        T.StructField(
            "self", T.StructType([T.StructField("href", T.StringType())])
        )
    ]
)

In [5]:
episode_image_schema = T.StructType(
    [
        T.StructField("medium", T.StringType()),
        T.StructField("original", T.StringType()),
    ]
)

In [6]:
episode_schema = T.StructType(
    [
        T.StructField("_links", episode_links_schema),
        T.StructField("airdate", T.DateType()),
        T.StructField("airstamp", T.TimestampType()),
        T.StructField("airtime", T.StringType()),
        T.StructField("id", T.StringType()),
        T.StructField("image", episode_image_schema),
        T.StructField("name", T.StringType()),
        T.StructField("number", T.LongType()),
        T.StructField("runtime", T.LongType()),
        T.StructField("season", T.LongType()),
        T.StructField("summary", T.StringType()),
        T.StructField("url", T.StringType()),
    ]
)

In [7]:
embedded_schema = T.StructType(
    [
        T.StructField(
            "_embedded",
            T.StructType(
                [
                    T.StructField(
                        "episodes", T.ArrayType(episode_schema)
                    )
                ]
            ),
        )
    ]
)

In [8]:
# 读取部分列
shows_with_schema = spark.read.json(
    "shows-silicon-valley.json",
    schema=embedded_schema, 
    mode="FAILFAST", 
)

In [9]:
shows_with_schema.printSchema()

root
 |-- _embedded: struct (nullable = true)
 |    |-- episodes: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _links: struct (nullable = true)
 |    |    |    |    |-- self: struct (nullable = true)
 |    |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- airdate: date (nullable = true)
 |    |    |    |-- airstamp: timestamp (nullable = true)
 |    |    |    |-- airtime: string (nullable = true)
 |    |    |    |-- id: string (nullable = true)
 |    |    |    |-- image: struct (nullable = true)
 |    |    |    |    |-- medium: string (nullable = true)
 |    |    |    |    |-- original: string (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- number: long (nullable = true)
 |    |    |    |-- runtime: long (nullable = true)
 |    |    |    |-- season: long (nullable = true)
 |    |    |    |-- summary: string (nullable = true)
 |    |    |    |-- url: string (nullable = t

In [10]:
# 查看两个包含时间戳的列airdate和airstamp的值
for column in ["airdate", "airstamp"]:
    shows_with_schema.select(f"_embedded.episodes.{column}").show()

+--------------------+
|             airdate|
+--------------------+
|[2014-04-06, 2014...|
+--------------------+

+--------------------+
|            airstamp|
+--------------------+
|[2014-04-07 10:00...|
+--------------------+



In [11]:
# 逐行输出两个包含时间戳的列airdate和airstamp的值
for column in ["airdate", "airstamp"]:
    shows_with_schema.select(f"_embedded.episodes.{column}").select(
        F.explode(column)
    ).show(5)

+----------+
|       col|
+----------+
|2014-04-06|
|2014-04-13|
|2014-04-20|
|2014-04-27|
|2014-05-04|
+----------+
only showing top 5 rows
+-------------------+
|                col|
+-------------------+
|2014-04-07 10:00:00|
|2014-04-14 10:00:00|
|2014-04-21 10:00:00|
|2014-04-28 10:00:00|
|2014-05-05 10:00:00|
+-------------------+
only showing top 5 rows


In [12]:
# 通过JSON获取schema
import json

other_shows_schema = T.StructType.fromJson(
    json.loads(shows_with_schema.schema.json())
)

In [13]:
print(other_shows_schema == shows_with_schema.schema)

True


In [14]:
# 以map形式显示episode的id和name
episode_name_id = shows_with_schema.select(
    F.map_from_arrays(
        F.col("_embedded.episodes.id"), F.col("_embedded.episodes.name")
    ).alias("name_id")
)

episode_name_id.show(truncate=120)

+------------------------------------------------------------------------------------------------------------------------+
|                                                                                                                 name_id|
+------------------------------------------------------------------------------------------------------------------------+
|{10897 -> Minimum Viable Product, 10898 -> The Cap Table, 10899 -> Articles of Incorporation, 10900 -> Fiduciary Duti...|
+------------------------------------------------------------------------------------------------------------------------+



## 4. Spark SQL Examples

### 4.1 case: 找出元素周期表中液体元素数量

In [15]:
# 读取元素周期表文件
elements = spark.read.csv(
    "Periodic_Table_Of_Elements.csv",
    header=True,
    inferSchema=True,
)

In [16]:
# 按period分组统计液态元素数量
elements.where(F.col("phase") == "liq").groupby("period").count().show()

+------+-----+
|period|count|
+------+-----+
|     6|    1|
|     4|    1|
+------+-----+



In [17]:
# 使用SQL查询按period分组统计液态元素数量
elements.createOrReplaceTempView("elements")

spark.sql(
    "select period, count(*) from elements where phase='liq' group by period"
).show()


26/01/19 16:22:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------+--------+
|period|count(1)|
+------+--------+
|     6|       1|
|     4|       1|
+------+--------+



### 管理视图 (Optional)

In [18]:
spark.catalog

In [19]:
spark.catalog.listTables()

[Table(name='elements', catalog=None, namespace=[], description=None, tableType='TEMPORARY', isTemporary=True)]

In [20]:
spark.catalog.dropTempView("elements")

True

In [21]:
spark.catalog.listTables()

[]

### 4.2 case: 通过Backblaze提供的硬盘信息和统计数据计算故障次数最多的硬盘型号

数据准备：将/shareddata下面的data_Q3_2019.zip解压到/data/lab05/backblaze目录下并删除非csv类型文件或目录

In [22]:
# 读取Backblaze硬盘数据
DATA_DIRECTORY = "/data/lab05/backblaze"

backblaze_2019 = spark.read.csv(
    DATA_DIRECTORY, header=True, inferSchema=True
)

In [23]:
backblaze_2019.printSchema()

root
 |-- date: date (nullable = true)
 |-- serial_number: string (nullable = true)
 |-- model: string (nullable = true)
 |-- capacity_bytes: long (nullable = true)
 |-- failure: integer (nullable = true)
 |-- smart_1_normalized: integer (nullable = true)
 |-- smart_1_raw: integer (nullable = true)
 |-- smart_2_normalized: integer (nullable = true)
 |-- smart_2_raw: integer (nullable = true)
 |-- smart_3_normalized: integer (nullable = true)
 |-- smart_3_raw: integer (nullable = true)
 |-- smart_4_normalized: integer (nullable = true)
 |-- smart_4_raw: integer (nullable = true)
 |-- smart_5_normalized: integer (nullable = true)
 |-- smart_5_raw: integer (nullable = true)
 |-- smart_7_normalized: integer (nullable = true)
 |-- smart_7_raw: long (nullable = true)
 |-- smart_8_normalized: integer (nullable = true)
 |-- smart_8_raw: integer (nullable = true)
 |-- smart_9_normalized: integer (nullable = true)
 |-- smart_9_raw: integer (nullable = true)
 |-- smart_10_normalized: integer (nul

In [24]:
# 将所有以smart开头的列转换为Long类型
backblaze_2019 = backblaze_2019.select(
    [
        F.col(x).cast(T.LongType()) if x.startswith("smart") else F.col(x)
        for x in backblaze_2019.columns
    ]
)

In [25]:
# 创建临时视图
backblaze_2019.createOrReplaceTempView("backblaze_stats_2019")

In [26]:
# 查询2019年发生故障的硬盘序列号（select和where用法举例）
spark.sql(
    "select serial_number from backblaze_stats_2019 where failure = 1"
).show(5)

+--------------+
| serial_number|
+--------------+
|      ZCH072D6|
|      ZCH06DMQ|
|      2AGMH2WY|
|S2ZYJ9CG107981|
|      ZCH09H7D|
+--------------+
only showing top 5 rows


In [27]:
# Todo: 用 DataFrame API 实现同样的查询
backblaze_2019.where("failure = 1").select(F.col("serial_number")).show(5)

+--------------+
| serial_number|
+--------------+
|      ZCH072D6|
|      ZCH06DMQ|
|      2AGMH2WY|
|S2ZYJ9CG107981|
|      ZCH09H7D|
+--------------+
only showing top 5 rows


In [28]:
# 查询不同容量的硬盘型号（groupby, having, order by用法举例）
spark.sql(
    """SELECT
           model,
           min(capacity_bytes / pow(1024, 3)) min_GB,
           max(capacity_bytes/ pow(1024, 3)) max_GB
        FROM backblaze_stats_2019
        GROUP BY 1
        HAVING min_GB != max_GB
        ORDER BY 3 DESC"""
).show(5)

+--------------------+--------------------+-----------------+
|               model|              min_GB|           max_GB|
+--------------------+--------------------+-----------------+
|       ST12000NM0007|-9.31322574615478...|          11176.0|
|HGST HUH721212ALN604|-9.31322574615478...|          11176.0|
|HGST HUH721010ALE600|-9.31322574615478...|           9314.0|
|       ST10000NM0086|-9.31322574615478...|           9314.0|
|        ST8000NM0055|-9.31322574615478...|7452.036460876465|
+--------------------+--------------------+-----------------+
only showing top 5 rows


In [ ]:
# Todo: 用DataFrame API实现同样的查询


+--------------------+--------------------+-----------------+
|               model|              min_GB|           max_GB|
+--------------------+--------------------+-----------------+
|       ST12000NM0007|-9.31322574615478...|          11176.0|
|HGST HUH721212ALN604|-9.31322574615478...|          11176.0|
|HGST HUH721010ALE600|-9.31322574615478...|           9314.0|
|       ST10000NM0086|-9.31322574615478...|           9314.0|
|        ST8000NM0055|-9.31322574615478...|7452.036460876465|
+--------------------+--------------------+-----------------+
only showing top 5 rows


开始计算故障次数最多的硬盘型号

In [30]:
# 创建临时视图（从此处开始计算故障次数最多的硬盘型号）
backblaze_2019.createOrReplaceTempView("drive_stats")

In [31]:
# 计算各型号硬盘运行总天数
spark.sql(
    """
    CREATE OR REPLACE TEMP VIEW drive_days AS
        SELECT model, count(*) AS drive_days
        FROM drive_stats
        GROUP BY model"""
)

DataFrame[]

In [ ]:
# Todo: 用 DataFrame API 实现同样的查询


In [33]:
# 计算各型号硬盘故障总天数
spark.sql(
    """CREATE OR REPLACE TEMP VIEW failures AS
           SELECT model, count(*) AS failures
           FROM drive_stats
           WHERE failure = 1
           GROUP BY model"""
)

DataFrame[]

In [ ]:
# Todo: 用 DataFrame API 实现同样的查询


In [35]:
# 计算各型号硬盘的故障率并按降序排序（完整代码）
spark.sql(
    """
    SELECT
        failures.model,
        failures / drive_days failure_rate
    FROM (
        SELECT
            model,
            count(*) AS drive_days
        FROM drive_stats
        GROUP BY model) drive_days
    INNER JOIN (
        SELECT
            model,
            count(*) AS failures
        FROM drive_stats
        WHERE failure = 1
        GROUP BY model) failures
    ON
        drive_days.model = failures.model
    ORDER BY 2 desc
    """
).show(5)

+------------------+--------------------+
|             model|        failure_rate|
+------------------+--------------------+
|     ST12000NM0117|0.019305019305019305|
|TOSHIBA MQ01ABF050|5.579360828423496E-4|
|       ST8000DM005|4.385964912280702E-4|
|        ST500LM030| 4.19639110365086E-4|
|     ST500LM012 HN|1.511585221015353...|
+------------------+--------------------+
only showing top 5 rows


In [36]:
# 计算各型号硬盘的故障率并按降序排序（优化后代码）
spark.sql(
    """
    WITH drive_days as (
        SELECT
            model,
            count(*) AS drive_days
        FROM drive_stats
        GROUP BY model),
    failures as (
        SELECT
            model,
            count(*) AS failures
        FROM drive_stats
        WHERE failure = 1
        GROUP BY model)
    SELECT
        failures.model,
        failures / drive_days failure_rate
    FROM drive_days
    INNER JOIN failures
    ON
        drive_days.model = failures.model
    ORDER BY 2 desc
    """
).show(5)

+------------------+--------------------+
|             model|        failure_rate|
+------------------+--------------------+
|     ST12000NM0117|0.019305019305019305|
|TOSHIBA MQ01ABF050|5.579360828423496E-4|
|       ST8000DM005|4.385964912280702E-4|
|        ST500LM030| 4.19639110365086E-4|
|     ST500LM012 HN|1.511585221015353...|
+------------------+--------------------+
only showing top 5 rows


In [ ]:
# Todo: 用DataFrame API实现计算各型号硬盘的故障率并按降序排序的函数


In [38]:
# 调用函数并显示结果
failure_rate(backblaze_2019).show(5)

+------------------+----------+--------+--------------------+
|             model|drive_days|failures|        failure_rate|
+------------------+----------+--------+--------------------+
|     ST12000NM0117|       259|       5|0.019305019305019305|
|TOSHIBA MQ01ABF050|     44808|      25|5.579360828423496E-4|
|       ST8000DM005|      2280|       1|4.385964912280702E-4|
|        ST500LM030|     21447|       9| 4.19639110365086E-4|
|     ST500LM012 HN|     46309|       7|1.511585221015353...|
+------------------+----------+--------+--------------------+
only showing top 5 rows


#### 编写一个函数，在给定容量（范围值）的情况下，根据故障率返回前三名最可靠的硬盘

In [39]:
full_data = backblaze_2019.selectExpr(
    "model", "capacity_bytes / pow(1024, 3) capacity_GB", "date", "failure"
)

In [40]:
drive_days = full_data.groupby("model", "capacity_GB").agg(
    F.count("*").alias("drive_days")
)

In [41]:
failures = (
    full_data.where("failure = 1")
    .groupby("model", "capacity_GB")
    .agg(F.count("*").alias("failures"))
)

In [42]:
summarized_data = (
    drive_days.join(failures, on=["model", "capacity_GB"], how="left")
    .fillna(0.0, ["failures"])
    .selectExpr("model", "capacity_GB", "failures / drive_days failure_rate")
    .cache()
)

In [43]:
def most_reliable_drive_for_capacity(data, capacity_GB=2048, precision=0.25, top_n=3):
    """Returns the top 3 drives for a given approximate capacity.

    Given a capacity in GB and a precision as a decimal number, we keep the N
    drives where:

    - the capacity is between (capacity * 1/(1+precision)), capacity * (1+precision)
    - the failure rate is the lowest

    """
    capacity_min = capacity_GB / (1 + precision)
    capacity_max = capacity_GB * (1 + precision)

    answer = (
        data.where(f"capacity_GB between {capacity_min} and {capacity_max}")
        .orderBy("failure_rate", "capacity_GB", ascending=[True, False])
        .limit(top_n)
    )

    return answer

In [44]:
most_reliable_drive_for_capacity(summarized_data, capacity_GB=11176.0).show()

+--------------------+-----------+--------------------+
|               model|capacity_GB|        failure_rate|
+--------------------+-----------+--------------------+
|HGST HUH721010ALE600|     9314.0|                 0.0|
|HGST HUH721212ALN604|    11176.0|1.585588480593982...|
|HGST HUH721212ALE600|    11176.0|1.636661211129296...|
+--------------------+-----------+--------------------+



### Exercise (Optional)
计算硬盘故障率通常采用MTBF (Mean Time Between Failures)，请按照新的计算方法借助大模型编写在给定容量（范围值）的情况下，根据故障率返回最可靠的硬盘的前10个

提示：每个硬盘可以由序列号、型号、容量三个唯一标识